# Policy Evaluation: DiD, Event Study, and SCM

This tutorial demonstrates how to evaluate policy effects using:
1. **Difference-in-Differences (DiD)** - Classic before/after comparison
2. **Event Study** - Dynamic effects over time
3. **Synthetic Control Method (SCM)** - Single treated unit

**Scenario**: Evaluating the effect of a minimum wage increase on employment.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../../src')

from causal_inference.did import did_2x2, event_study
from causal_inference.scm import synthetic_control

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Generate Panel Data

We simulate a panel of 50 states over 20 time periods.
- Treatment: Minimum wage increase in period 10
- Treated states: First 10 states
- True treatment effect: -2.0 (negative employment effect)

In [ ]:
# Parameters
n_units = 50
n_periods = 20
treatment_period = 10
n_treated = 10
true_effect = -2.0

# Generate data
unit_ids = np.repeat(np.arange(n_units), n_periods)
time_ids = np.tile(np.arange(n_periods), n_units)

# Unit fixed effects
unit_fe = np.repeat(np.random.randn(n_units) * 3, n_periods)

# Time trend (common)
time_trend = np.tile(np.linspace(0, 5, n_periods), n_units)

# Treatment indicator
treated = (unit_ids < n_treated) & (time_ids >= treatment_period)

# Outcome
Y = 50 + unit_fe + time_trend + true_effect * treated + np.random.randn(len(treated)) * 2

# Create DataFrame
df = pd.DataFrame({
    'unit': unit_ids,
    'time': time_ids,
    'Y': Y,
    'treated_unit': unit_ids < n_treated,
    'post': time_ids >= treatment_period,
    'treatment': treated
})

print(f"Panel dimensions: {n_units} units x {n_periods} periods")
print(f"Treatment: {n_treated} units treated at period {treatment_period}")
print(f"True effect: {true_effect}")
df.head(10)

## Visualize Pre/Post Trends

In [ ]:
# Calculate group means
means = df.groupby(['time', 'treated_unit'])['Y'].mean().unstack()

plt.figure(figsize=(10, 6))
plt.plot(means.index, means[False], 'b-o', label='Control', markersize=5)
plt.plot(means.index, means[True], 'r-o', label='Treated', markersize=5)
plt.axvline(x=treatment_period - 0.5, color='gray', linestyle='--', label='Treatment')
plt.xlabel('Time Period')
plt.ylabel('Outcome (Employment)')
plt.title('Parallel Trends: Treated vs Control Units')
plt.legend()
plt.show()

---

## Method 1: Classic Difference-in-Differences

The DiD estimator compares changes over time between treated and control groups:

$$\hat{\tau}_{DiD} = (\bar{Y}_{treated,post} - \bar{Y}_{treated,pre}) - (\bar{Y}_{control,post} - \bar{Y}_{control,pre})$$

In [ ]:
# Run DiD
result_did = did_2x2(
    outcome=df['Y'].values,
    treatment=df['treated_unit'].astype(int).values,
    post=df['post'].astype(int).values,
    cluster=df['unit'].values  # Cluster SE at unit level
)

print("=== Difference-in-Differences Results ===")
print(f"Estimate: {result_did['estimate']:.3f}")
print(f"Std Error: {result_did['se']:.3f}")
print(f"95% CI: [{result_did['ci_lower']:.3f}, {result_did['ci_upper']:.3f}]")
print(f"p-value: {result_did['p_value']:.4f}")
print(f"\nTrue effect: {true_effect}")
print(f"Bias: {result_did['estimate'] - true_effect:.3f}")

### DiD Interpretation

- **Estimate**: The average treatment effect on the treated (ATT)
- **Key assumption**: Parallel trends - treated and control would have trended similarly absent treatment
- **Cluster SE**: We cluster at the unit level because errors are correlated within units

---

## Method 2: Event Study Design

Event study estimates dynamic effects by period relative to treatment.

Key outputs:
- **Pre-period coefficients**: Should be ~0 if parallel trends holds
- **Post-period coefficients**: Treatment effect over time

In [ ]:
# Create relative time variable
df['rel_time'] = df['time'] - treatment_period
df['rel_time'] = df['rel_time'].clip(-5, 9)  # Bin endpoints

# Run event study
result_es = event_study(
    outcome=df['Y'].values,
    unit=df['unit'].values,
    time=df['time'].values,
    treated_unit=df['treated_unit'].values,
    treatment_time=np.where(df['treated_unit'], treatment_period, np.inf),
    n_leads=5,
    n_lags=9
)

print("=== Event Study Results ===")
print(f"Pre-trends p-value: {result_es['pretrends_pvalue']:.4f}")
print(f"  (p > 0.05 suggests parallel trends satisfied)\n")

print("Coefficients by period:")
for i, (coef, se) in enumerate(zip(result_es['coefficients'], result_es['ses'])):
    rel_t = i - 5  # Relative time
    marker = '***' if abs(coef/se) > 2.58 else ('**' if abs(coef/se) > 1.96 else '')
    print(f"  t={rel_t:+3d}: {coef:7.3f} ({se:.3f}) {marker}")

In [ ]:
# Plot event study
rel_times = np.arange(-5, 10)
coefs = result_es['coefficients']
ses = result_es['ses']

plt.figure(figsize=(10, 6))
plt.errorbar(rel_times, coefs, yerr=1.96*np.array(ses), fmt='o-', capsize=3)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
plt.axvline(x=-0.5, color='red', linestyle='--', label='Treatment')
plt.axhline(y=true_effect, color='green', linestyle=':', label=f'True effect ({true_effect})')
plt.xlabel('Periods Relative to Treatment')
plt.ylabel('Effect on Employment')
plt.title('Event Study: Dynamic Treatment Effects')
plt.legend()
plt.show()

### Event Study Interpretation

- **Pre-period coefficients near 0**: Supports parallel trends assumption
- **Jump at t=0**: Immediate treatment effect
- **Post-period pattern**: Shows how effect evolves over time
- **Pre-trends test**: Joint F-test that all pre-period coefficients = 0

---

## Method 3: Synthetic Control Method

SCM creates a synthetic "control" as a weighted average of untreated units.

Best when:
- Single treated unit (or small number)
- Long pre-treatment period
- Good donor pool of controls

In [ ]:
# Reshape data for SCM (units x time)
outcomes_matrix = df.pivot(index='unit', columns='time', values='Y').values

# Treatment indicator (first unit is treated)
treatment_indicator = np.zeros(n_units)
treatment_indicator[0] = 1  # Only first unit for SCM demo

# Run SCM
result_scm = synthetic_control(
    outcomes=outcomes_matrix,
    treatment=treatment_indicator,
    treatment_period=treatment_period,
    n_placebo=20  # Placebo inference
)

print("=== Synthetic Control Results ===")
print(f"ATT Estimate: {result_scm['estimate']:.3f}")
print(f"p-value (placebo): {result_scm['p_value']:.4f}")
print(f"Pre-treatment RMSPE: {result_scm['pre_rmspe']:.3f}")
print(f"\nTop donor weights:")
for idx, weight in sorted(enumerate(result_scm['weights']), key=lambda x: -x[1])[:5]:
    if weight > 0.01:
        print(f"  Unit {idx}: {weight:.3f}")

In [ ]:
# Plot SCM
treated_outcome = outcomes_matrix[0, :]
synthetic_outcome = result_scm['synthetic']

plt.figure(figsize=(10, 6))
plt.plot(range(n_periods), treated_outcome, 'b-o', label='Treated Unit', markersize=5)
plt.plot(range(n_periods), synthetic_outcome, 'r--s', label='Synthetic Control', markersize=5)
plt.axvline(x=treatment_period - 0.5, color='gray', linestyle='--')
plt.xlabel('Time Period')
plt.ylabel('Outcome')
plt.title('Synthetic Control: Treated vs Synthetic')
plt.legend()
plt.show()

### SCM Interpretation

- **Pre-treatment fit**: Synthetic should match treated closely before treatment
- **Post-treatment gap**: Difference = treatment effect
- **Placebo inference**: Permute treatment across donors; p-value = proportion with larger gaps
- **Weights**: Show which donors matter most

---

## Method Comparison

In [ ]:
print("=== Comparison of Methods ===")
print(f"True Effect: {true_effect}")
print(f"")
print(f"{'Method':<20} {'Estimate':>10} {'SE':>10} {'Bias':>10}")
print("-" * 50)
print(f"{'DiD':<20} {result_did['estimate']:>10.3f} {result_did['se']:>10.3f} {result_did['estimate'] - true_effect:>10.3f}")
print(f"{'Event Study (avg)':<20} {np.mean(coefs[5:]):>10.3f} {'-':>10} {np.mean(coefs[5:]) - true_effect:>10.3f}")
print(f"{'Synthetic Control':<20} {result_scm['estimate']:>10.3f} {'-':>10} {result_scm['estimate'] - true_effect:>10.3f}")

---

## Summary: When to Use Each Method

| Method | Best For | Key Requirement |
|--------|----------|----------------|
| **DiD** | Many treated/control units | Parallel trends |
| **Event Study** | Dynamic effects, pre-trends testing | Multiple pre-periods |
| **SCM** | Single treated unit | Good donor pool, long pre-period |

### Key Takeaways

1. **Always check parallel trends** before trusting DiD
2. **Event study pre-periods** provide visual and statistical test of parallel trends
3. **SCM works best** when you have a single treated unit and need interpretable counterfactual
4. **Cluster standard errors** at the level of treatment assignment (usually unit)

---

*Created: Session 98 - Causal Inference Mastery*